##  Utforska och fördjupa er i neurala nätverk

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# --------------------------------------------
# 1. Load dataset
# --------------------------------------------

(X_train, y_train), (X_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Combine train and test data
X = np.concatenate([X_train, X_test])
y = np.concatenate([y_train, y_test])

# --------------------------------------------
# 2. Explore dataset
# --------------------------------------------

print("Dataset shape:")
print("X shape:", X.shape)
print("y shape:", y.shape)

# Class names
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

print("\nClasses in dataset:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

# Class distribution
class_distribution = pd.Series(y).value_counts().sort_index()

print("\nClass distribution:")
print(class_distribution)

# --------------------------------------------
# 3. Show sample image
# --------------------------------------------

plt.imshow(X[2], cmap="gray")
plt.title(f"Class: {class_names[y[2]]}")
plt.axis("off")
plt.show()



## Resultat 
- Datasetet består av 70 000 bilder från Fashion-MNIST.
- Datasetet innehåller totalt tio klasser, representerade av etiketterna 0–9. Klassfördelningen visar att varje klass innehåller exakt 7 000 bilder.
- Detta innebär att datasetet är balanserat och att det inte finns någon klassobalans som riskerar att påverka modellträningen negativt.

In [ ]:
# --------------------------------------------
# 4. Train/Test split
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# --------------------------------------------
# Validation split
# --------------------------------------------

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.125,
    random_state=42,
    stratify=y_train
)

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Val shape:", X_val.shape)

# --------------------------------------------
# 5. Normalize images
# --------------------------------------------

X_train = X_train / 255.0
X_test = X_test / 255.0
X_val = X_val / 255.0

print("\nPixel range after normalization:")
print("Min:", X_train.min())
print("Max:", X_train.max())

# --------------------------------------------
# 6A. Reshape for Dense model
# --------------------------------------------

X_train_dense = X_train.reshape(-1, 784)
X_test_dense = X_test.reshape(-1, 784)
X_val_dense = X_val.reshape(-1, 784)

print("\nDense model shape:")
print("X_train_dense:", X_train_dense.shape)
print("X_test_dense:", X_test_dense.shape)
print("X_val_dense:", X_val_dense.shape)

# --------------------------------------------
# 6B. Reshape for CNN model
# --------------------------------------------

X_train_cnn = X_train.reshape(-1, 28, 28, 1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)
X_val_cnn = X_val.reshape(-1, 28, 28, 1)

print("\nCNN model shape:")
print("X_train_cnn:", X_train_cnn.shape)
print("X_test_cnn:", X_test_cnn.shape)
print("X_val_cnn:", X_val_cnn.shape)

In [ ]:
## First model 

base_model = keras.Sequential([
    # Input lager
    layers.Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=(28,28,1)
    ),

    layers.MaxPooling2D((2,2)),

    layers.Conv2D(
        64,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D((2,2)),

    layers.Conv2D(
        128,
        (3,3),
        activation="relu"
    ),

    layers.MaxPooling2D((2,2)),

    # Gör om till 1D
    layers.Flatten(),

    # Dense-lager
    layers.Dense(128, activation="relu"),

    # Dropout lager
    layers.Dropout(0.5),

    # Outputlager
    layers.Dense(10, activation="softmax")
])

base_model.summary()

In [ ]:
## Säkerställer att modellen fungerar 

base_model.compile(
    optimizer = "adam",
    loss = "sparse_categorical_crossentropy",
    metrics = ["accuracy"]
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = base_model.fit(
    X_train_cnn,
    y_train,
    epochs=10,
    validation_data=(X_val_cnn, y_val),
    callbacks=[early_stop]
)

In [ ]:
# Funktion för att plotta loss och accuracy. Ange history och vad du vill ha för titel i plotten =)
def plot_training_history(history, model_name="Grundmodell"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(12, 5))

    # Plotta Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(epochs, acc, 'b', label='Training accuracy')
    plt.plot(epochs, val_acc, 'r', label='Validation accuracy')
    plt.title(f'{model_name} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.xticks(epochs)
    plt.legend()

    # Plotta Loss
    plt.subplot(1, 2, 2)
    plt.plot(epochs, loss, 'b', label='Training loss')
    plt.plot(epochs, val_loss, 'r', label='Validation loss')
    plt.title(f'{model_name} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.xticks(epochs)
    plt.legend()

    plt.show()

# Anropar funktionen med den tidigare historyn
plot_training_history(history, "base_model") 

In [ ]:
# Steg 5 Testa och förbättra modellen

# Modell Variant 1
model_fewer_filters = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(16, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_fewer_filters.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_fewer_filters = model_fewer_filters.fit(
    X_train_cnn,
    y_train,
    epochs=10,
    validation_data=(X_val_cnn, y_val),
)

test_loss_fewer, test_acc_fewer = model_fewer_filters.evaluate(X_test_cnn, y_test)

In [ ]:
# Modell Variant 2

model_more_filters = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_more_filters.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_more_filters = model_more_filters.fit(
    X_train_cnn,
    y_train,
    epochs=10,
    validation_data=(X_val_cnn, y_val),
)

test_loss_more, test_acc_more = model_more_filters.evaluate(X_test_cnn, y_test)

In [ ]:
# Modell Variant 3

model_extra_layer = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model_extra_layer.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_extra_layer = model_extra_layer.fit(
    X_train_cnn,
    y_train,
    epochs=10,
    validation_data=(X_val_cnn, y_val),
)

test_loss_extra, test_acc_extra = model_extra_layer.evaluate(X_test_cnn, y_test)

In [ ]:
base_test_loss, base_test_acc = base_model.evaluate(X_test_cnn, y_test)

In [ ]:
results = pd.DataFrame({
    'Modell': [
        'Grundmodell',
        'Färre filter',
        'Fler filter',
        'Extra Conv2D-lager'
    ],
    'Test accuracy': [
        base_test_acc,
        test_acc_fewer,
        test_acc_more,
        test_acc_extra
    ],
    'Test loss': [
        base_test_loss,
        test_loss_fewer,
        test_loss_more,
        test_loss_extra
    ]
})

results

In [ ]:
# Grafer för modellvarianterna 1,2 och 3
plot_training_history(history_fewer_filters, 'Modell 1 - Färre filter')
plot_training_history(history_more_filters, 'Modell 2 - Fler filter')
plot_training_history(history_extra_layer, 'Modell 3 - Extra Conv2D-lager')